# Per-Bin Decay Rate Tuning for User+WC MoE

Tunes the time-decay rate independently for each wallclock bin and power user group.
Different clusters may have different optimal decay rates:
- High-volume short-job bins (<=2h) change fast → may benefit from higher decay
- Low-volume long-job bins (>48h) change slowly → may need flat weighting

**Approach:** For each bin, test multiple decay rates and pick the best.
Then combine the per-bin-optimal models for the final aggregate MAE.

**Dataset:** NLR Kestrel, expanded window (2025-01-01 to 2025-06-26, ~193 days, 2.7M rows)
**Model:** XGBoost Tuned (200 trees, depth 12)
**Lookback:** 120 days
**Rolling eval:** 120 windows × 6h

## 1. Setup

In [ ]:
from datetime import datetime, timedelta, timezone
from pathlib import Path
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq

from hpc_oda_commons.models.experimental.xgboost_tuned_model import (
    ExperimentalXGBoostTunedConfig, ExperimentalXGBoostTunedModel,
)

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})

REPO_ROOT = Path.cwd().parent.parent
DATA_PATH = REPO_ROOT / 'workspace' / 'data' / 'datasets' / 'nlr_kestrel' / 'data.parquet'

In [ ]:
# Load with enough history for 120-day lookback
table = pq.read_table(DATA_PATH)
lo = datetime(2025, 1, 1, tzinfo=timezone.utc)
hi = datetime(2025, 6, 26, tzinfo=timezone.utc) + timedelta(days=1)
sc = table.column('submit_time')
ec = table.column('end_time')
mask = pc.and_(
    pc.less(sc, pa.scalar(hi, type=sc.type)),
    pc.greater_equal(ec, pa.scalar(lo, type=ec.type)),
)
df = table.filter(mask).to_pandas()
rows_all = df.to_dict('records')
print(f'Loaded: {len(rows_all):,} rows')
print(f'Date range: {df["submit_time"].min().date()} to {df["submit_time"].max().date()}')
span = (df['submit_time'].max() - df['submit_time'].min()).days
print(f'Span: {span} days')

## 2. Configuration

In [ ]:
N_WINDOWS = 120
TEST_WINDOW_HOURS = 6
TRAINING_LOOKBACK_DAYS = 120

DECAY_RATES = [0.0, 0.01, 0.03, 0.05, 0.1]

POWER_USER_PERCENTILE = 0.99

BIN_EDGES_H = [0, 2, 4, 24, 48, float('inf')]
BIN_LABELS = ['<=2h', '2-4h', '4-24h', '24-48h', '>48h']

def make_config(decay_rate):
    return ExperimentalXGBoostTunedConfig(
        n_windows=N_WINDOWS,
        test_window_hours=TEST_WINDOW_HOURS,
        training_lookback_days=TRAINING_LOOKBACK_DAYS,
        max_svd_components=256,
        target_max_one_hot_width=2048,
        random_state=42,
        n_estimators=200,
        max_depth=12,
        learning_rate=0.03,
        min_child_weight=5,
        gamma=0.1,
        time_decay_rate=decay_rate,
    )

print(f'Windows: {N_WINDOWS} x {TEST_WINDOW_HOURS}h = {N_WINDOWS*TEST_WINDOW_HOURS/24:.0f} days test')
print(f'Lookback: {TRAINING_LOOKBACK_DAYS} days')
print(f'Decay rates: {DECAY_RATES}')
print(f'Bins: {BIN_LABELS}')

## 3. Split data into bins

In [ ]:
def assign_bin(row):
    wc_h = (row.get('requested_seconds') or 0) / 3600
    for i in range(len(BIN_EDGES_H) - 1):
        if wc_h <= BIN_EDGES_H[i + 1]:
            return BIN_LABELS[i]
    return BIN_LABELS[-1]

# Identify power users
user_counts = Counter(r.get('user') for r in rows_all)
threshold = np.percentile(list(user_counts.values()), POWER_USER_PERCENTILE * 100)
power_users = {u for u, c in user_counts.items() if c >= threshold}

# Split into groups: power users per bin, non-power users per bin
groups = {}  # (group_label) -> rows

for row in rows_all:
    user = row.get('user')
    bl = assign_bin(row)
    if user in power_users:
        key = f'power:{user[:7]}/{bl}'
    else:
        key = f'non-power/{bl}'
    groups.setdefault(key, []).append(row)

# Filter to groups with enough data
valid_groups = {k: v for k, v in groups.items() if len(v) >= 100}

print(f'Power users: {len(power_users)}')
print(f'Total groups: {len(groups)}')
print(f'Valid groups (>= 100 rows): {len(valid_groups)}')
print(f'\n{"Group":<30} {"Rows":>10}')
print('-' * 42)
for k in sorted(valid_groups.keys()):
    print(f'{k:<30} {len(valid_groups[k]):>10,}')

## 4. Tune decay rate per bin

For each group, test all decay rates and find the best one.

In [ ]:
bin_results = {}  # group -> {rate: {mae, rmse, scored}}
bin_best = {}     # group -> (best_rate, best_mae)

for group_name, group_rows in sorted(valid_groups.items()):
    print(f'\n{"="*60}')
    print(f'Group: {group_name} ({len(group_rows):,} rows)')
    print(f'{"="*60}')
    
    bin_results[group_name] = {}
    
    for rate in DECAY_RATES:
        try:
            config = make_config(rate)
            model = ExperimentalXGBoostTunedModel(config)
            payload = model.evaluate(group_rows, capture_artifacts=True)
            scored = payload['summary']['rows_scored']
            if scored > 0:
                bin_results[group_name][rate] = {
                    'mae': payload['mae'],
                    'rmse': payload['rmse'],
                    'scored': scored,
                    'y_true': payload.get('_y_true', []),
                    'y_pred': payload.get('_y_pred', []),
                }
                print(f'  rate={rate:.2f}: MAE={payload["mae"]:>10,.0f}s  scored={scored:,}')
            else:
                print(f'  rate={rate:.2f}: 0 scored')
        except Exception as e:
            print(f'  rate={rate:.2f}: FAILED — {e}')
    
    # Find best rate for this group
    if bin_results[group_name]:
        best_rate = min(bin_results[group_name].items(), key=lambda x: x[1]['mae'])
        bin_best[group_name] = (best_rate[0], best_rate[1]['mae'])
        print(f'  BEST: rate={best_rate[0]:.2f}, MAE={best_rate[1]["mae"]:,.0f}s')

## 5. Results: Best decay rate per group

In [ ]:
print('=' * 70)
print('BEST DECAY RATE PER GROUP')
print('=' * 70)

print(f'\n{"Group":<30} {"Best rate":>10} {"Best MAE":>10} {"Flat MAE":>10} {"Improvement":>12}')
print('-' * 75)
for group_name in sorted(bin_best.keys()):
    best_rate, best_mae = bin_best[group_name]
    flat_mae = bin_results[group_name].get(0.0, {}).get('mae', best_mae)
    improvement = (best_mae - flat_mae) / flat_mae * 100 if flat_mae > 0 else 0
    print(f'{group_name:<30} {best_rate:>10.2f} {best_mae:>10,.0f}s {flat_mae:>10,.0f}s {improvement:>+11.1f}%')

## 6. Aggregate: per-bin-optimal vs flat

In [ ]:
# Combine predictions using each group's best rate
optimal_true = []
optimal_pred = []
flat_true = []
flat_pred = []

for group_name in bin_best:
    best_rate = bin_best[group_name][0]
    # Best-rate predictions
    best_data = bin_results[group_name].get(best_rate, {})
    if best_data.get('y_true'):
        optimal_true.extend(best_data['y_true'])
        optimal_pred.extend(best_data['y_pred'])
    # Flat-rate predictions
    flat_data = bin_results[group_name].get(0.0, {})
    if flat_data.get('y_true'):
        flat_true.extend(flat_data['y_true'])
        flat_pred.extend(flat_data['y_pred'])

if optimal_true and flat_true:
    opt_mae = np.mean(np.abs(np.array(optimal_true) - np.array(optimal_pred)))
    opt_rmse = np.sqrt(np.mean((np.array(optimal_true) - np.array(optimal_pred))**2))
    flat_mae = np.mean(np.abs(np.array(flat_true) - np.array(flat_pred)))
    flat_rmse = np.sqrt(np.mean((np.array(flat_true) - np.array(flat_pred))**2))
    
    print('AGGREGATE COMPARISON')
    print('=' * 60)
    print(f'\n{"Approach":<35} {"MAE":>10} {"RMSE":>10} {"Scored":>10}')
    print('-' * 70)
    print(f'{"Flat weighting (rate=0 everywhere)":<35} {flat_mae:>10,.0f}s {flat_rmse:>10,.0f}s {len(flat_true):>10,}')
    print(f'{"Per-bin optimal decay rate":<35} {opt_mae:>10,.0f}s {opt_rmse:>10,.0f}s {len(optimal_true):>10,}')
    print(f'\nImprovement: {(opt_mae - flat_mae) / flat_mae * 100:+.1f}% MAE')
else:
    print('Could not compute aggregate — missing predictions')

In [ ]:
# Visualize best decay rate per group
groups_sorted = sorted(bin_best.keys())
best_rates = [bin_best[g][0] for g in groups_sorted]

fig, ax = plt.subplots(figsize=(12, 5))
colors = ['seagreen' if 'power' in g else 'steelblue' for g in groups_sorted]
bars = ax.barh(groups_sorted[::-1], best_rates[::-1], color=colors[::-1])
ax.set_xlabel('Optimal decay rate')
ax.set_title('Best Time-Decay Rate Per Group\n(0 = flat weighting, higher = more recency bias)')

for bar, rate in zip(bars, best_rates[::-1]):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
            f'{rate:.2f}', va='center', fontsize=9)

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color='seagreen', label='Power user bins'),
    Patch(color='steelblue', label='Non-power user bins'),
])
plt.tight_layout()
plt.show()

## 7. Conclusions

In [ ]:
print('CONCLUSIONS')
print('=' * 60)
print(f'\nGroups analyzed: {len(bin_best)}')
print(f'\nDecay rate distribution across groups:')
rate_counts = Counter(bin_best[g][0] for g in bin_best)
for rate in sorted(rate_counts.keys()):
    print(f'  rate={rate:.2f}: {rate_counts[rate]} groups')

# Is there a pattern? Do short-job bins prefer higher decay?
print(f'\nPattern check — do short-job bins prefer higher decay?')
for bl in BIN_LABELS:
    matching = [g for g in bin_best if bl in g]
    if matching:
        rates = [bin_best[g][0] for g in matching]
        print(f'  {bl} bins: mean optimal rate = {np.mean(rates):.3f} (n={len(rates)})')

if optimal_true and flat_true:
    improvement = (opt_mae - flat_mae) / flat_mae * 100
    print(f'\nPer-bin optimal vs flat: {improvement:+.1f}% MAE')
    if improvement < -2:
        print('Per-bin decay tuning provides meaningful improvement.')
        print('Different clusters benefit from different decay rates.')
    elif improvement > 2:
        print('Per-bin tuning hurts — flat weighting is better overall.')
        print('The per-bin optimization may be overfitting to the evaluation period.')
    else:
        print('Minimal difference — decay rate does not significantly affect results.')
        print('Flat weighting is simpler and works just as well.')